# Stoa 04 — bias slicing plots

Draws the figures for `docs/bias/bias_slicing.md` from the team's bias report. The slices themselves are defined in one place only, `src/bias/slices.py` with settings in `configs/bias.yaml`; this notebook reads the report they produce and doesn't redefine any slice. To change a slice or a cut-off, change the config (in a PR with a test), rebuild the report and re-run this notebook.

**Before running** (locally: from the repo root, in the project environment; in Colab, the first cell does this for you):

```bash
.venv/bin/dvc pull                                   # the report is versioned with the data
.venv/bin/python -m src.bias.representation          # or rebuild it: step 1 (~35 s)
.venv/bin/python -m src.bias.slice_errors            # steps 2-3 (~18 min)
```

| Reads (`data/processed/bias/melbourne/`) | |
|---|---|
| `slice_representation.csv` | one row per slice group: rows, sensors, shares, counts, ratio against a reference |
| `slice_crosstabs.csv` | time of day × rain, precinct × era |
| `coverage_by_year.csv` | rows, sensors and pedestrians per year |
| `slice_errors.csv` | one row per mitigation and slice group: the baseline's WAPE on held-out sensors, its 95% interval, `worse` |
| `mitigation_summary.csv` | one row per mitigation: overall WAPE, bias, MAE |

Every number comes from these files; the notebook only draws them (no metric or slice is computed here), so the figures can't drift from the doc.

Writes static PNGs to `docs/bias/figures/` (light background, about 50–200 KB each); the Colab version writes them to `MyDrive/stoa/bias_figures/`.

No people-level fields exist in this data, by design (no personal data, `CLAUDE.md` core rule 6): the groups are places, times and conditions. Slices are for checking, never model features (core rule 9).

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from matplotlib.colors import LinearSegmentedColormap

# Run from notebooks/ or the repo root.
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
REPORT = ROOT / "data" / "processed" / "bias" / "melbourne"
FIGURES = ROOT / "docs" / "bias" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

needed = ["slice_representation.csv", "slice_crosstabs.csv", "coverage_by_year.csv",
          "slice_errors.csv", "mitigation_summary.csv"]
missing = [f for f in needed if not (REPORT / f).exists()]
if missing:
    raise FileNotFoundError(f"missing {missing} in {REPORT}: run `.venv/bin/dvc pull` or "
                            "`.venv/bin/python -m src.bias.representation` and `src.bias.slice_errors` from the repo root")

rep = pd.read_csv(REPORT / "slice_representation.csv")
cross = pd.read_csv(REPORT / "slice_crosstabs.csv")
years = pd.read_csv(REPORT / "coverage_by_year.csv")
errors = pd.read_csv(REPORT / "slice_errors.csv")
summary = pd.read_csv(REPORT / "mitigation_summary.csv").set_index("mitigation")
CONFIG = ROOT / "configs" / "bias.yaml"
cfg = yaml.safe_load(CONFIG.read_text())
print(f"{len(rep)} slice groups, {len(cross)} cross-table cells, {len(years)} years; figures -> {FIGURES}")

### Chart style

Light surface, recessive axes, hairline grid, text in ink colours (never the series colour). Two series at most per chart: blue for the data, orange for the reference it's compared with; both validated for colour-vision deficiency, and every chart also has a legend or direct labels so colour never carries meaning alone.

In [ ]:
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
DATA, REFERENCE = "#2a78d6", "#eb6834"   # validated categorical slots 1 and 2
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 10, "text.color": INK,
    "axes.facecolor": SURFACE, "figure.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False,
})

PRETTY = {   # readable group names (report values -> labels)
    "night": "Night 0–6", "morning_peak": "Morning peak 6–10", "midday": "Midday 10–16",
    "evening_peak": "Evening peak 16–20", "late_evening": "Late evening 20–24",
    "pre_covid": "Pre-COVID", "covid": "COVID", "recovery": "Recovery", "post_metro_tunnel": "Post Metro Tunnel",
    "dry": "Dry (<0.1 mm)", "light": "Light (0.1–1 mm)", "rain": "Rain (≥1 mm)",
    "cold": "Cold (<10 °C)", "mild": "Mild (10–25 °C)", "warm": "Warm (25–30 °C)", "hot": "Hot (≥30 °C)",
    "not_holiday": "Not a holiday", "holiday": "Public holiday", "school_holidays": "School holidays", "term": "School term",
    "not_teaching": "Not uni teaching", "teaching": "Uni teaching", "weekday": "Weekday", "weekend": "Weekend",
    "none": "No AFL match", "marvel": "AFL at Marvel", "mcg": "AFL at the MCG", "both": "AFL at both",
    "quiet": "Quiet", "medium": "Medium", "busy": "Busy", "low": "Low", "high": "High",
    "auto_ok": "Matched automatically", "needs_review": "Match needs review",
    "training_area": "Training area", "unimelb_edge": "UniMelb edge", "carlton": "Carlton", "north_melbourne": "North Melbourne",
    "archive": "Archive", "live": "Live", "wayback": "Wayback",
}
pretty = lambda g: PRETTY.get(g, g)


def grid_on(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def save(fig, name, dpi=150):
    path = FIGURES / name
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor=SURFACE)
    plt.close(fig)
    print(f"saved {path.name} ({path.stat().st_size / 1e3:.0f} KB)")


def slice_rows(dimension, order=None):
    d = rep[(rep.dimension == dimension) & ~rep.group.isin(["unknown", "no_location"])].copy()
    if order:
        d = d.set_index("group").loc[[g for g in order if g in set(d.group)]].reset_index()
    return d

## 1. Precincts: share of sensors vs share of the walkable network

The main finding. Each precinct's share of the located sensors next to its share of the OSM walkable network (the report's reference). The number at the end of each pair is the ratio: above 1, more sensors than the precinct's share of the network; below 1, fewer. Sensor counts are in the labels, because conclusions from 2–3 sensors are weak.

In [ ]:
p = slice_rows("precinct")
p["sensor_share"] = p.n_sensors / p.n_sensors.sum()   # share of located sensors (the report's ratio uses this)
p = p.sort_values("ratio")
y = np.arange(len(p))
h = 0.38

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(y + h / 2, p.sensor_share, h, color=DATA, edgecolor=SURFACE, linewidth=1, label="Share of sensors")
ax.barh(y - h / 2, p.reference_share, h, color=REFERENCE, edgecolor=SURFACE, linewidth=1,
        label="Share of walkable network")
for yi, r in zip(y, p.itertuples()):
    ax.text(max(r.sensor_share, r.reference_share) + 0.01, yi, f"×{r.ratio:.2f}", va="center", color=INK, fontsize=9)
ax.set_yticks(y, [f"{g}  ({n})" for g, n in zip(p.group, p.n_sensors)])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlim(0, p[["sensor_share", "reference_share"]].max().max() * 1.15)
grid_on(ax, "x")
ax.set_title("Share of sensors against share of the walkable network, by precinct", pad=22)
ax.text(0, 1.015, "Precinct (number of sensors); ratio = share of sensors ÷ share of network",
        transform=ax.transAxes, color=INK2, fontsize=9)
ax.legend(loc="lower right")
save(fig, "precinct_sensors_vs_network.png")

## 2. Time, calendar, weather and era: ratio against the hours in the study period

Each dot is one group's share of the data divided by its share of all hours in the study period. 1 means the group has exactly its fair share; the shaded band is ±25%. Log scale, so 0.5 and 2 are the same distance from 1. Groups outside the band are labelled.

In [ ]:
DIMENSIONS = [
    ("time_of_day", "Time of day", list(cfg["time_of_day"])),
    ("day_type", "Day", ["weekday", "weekend"]),
    ("public_holiday", "Public holiday", ["not_holiday", "holiday"]),
    ("school_term", "School term", ["term", "school_holidays"]),
    ("uni_teaching", "University", ["teaching", "not_teaching"]),
    ("rain", "Rain", list(cfg["rain_mm"]["labels"])),
    ("temperature", "Temperature", list(cfg["temperature_c"]["labels"])),
    ("afl_day", "AFL match day", ["none", "marvel", "mcg", "both"]),
    ("era", "Era", list(cfg["eras"])),
]
rows, ticks = [], []   # a heading row (None) before each slice's groups
for dim, title, order in DIMENSIONS:
    d = slice_rows(dim, order).dropna(subset=["ratio"])
    rows.append(None)
    ticks.append(title)
    rows.extend(d.itertuples())
    ticks.extend(pretty(g) for g in d.group)
y = np.arange(len(rows))[::-1]

fig, ax = plt.subplots(figsize=(8, 9))
ax.axvspan(0.8, 1.25, color=GRID, alpha=0.6, linewidth=0)
ax.axvline(1, color=AXIS, linewidth=1.5)
for yi, r in zip(y, rows):
    if r is None:
        continue
    outside = not 0.8 <= r.ratio <= 1.25
    ax.plot(r.ratio, yi, "o", ms=8, color=REFERENCE if outside else DATA, markeredgecolor=SURFACE, markeredgewidth=1.5)
    if outside:
        ax.text(r.ratio * 1.06, yi, f"{r.ratio:.2f}", va="center", fontsize=9, color=INK)
ax.set_xscale("log")
ax.set_xticks([0.5, 0.8, 1, 1.25, 2], ["0.5", "0.8", "1", "1.25", "2"])
ax.minorticks_off()
ax.set_xlim(0.45, 3)
ax.set_yticks(y, ticks)
for label, r in zip(ax.get_yticklabels(), rows):
    if r is None:   # slice headings: bold, in primary ink
        label.set_fontweight("bold")
        label.set_color(INK)
ax.tick_params(axis="y", length=0)
grid_on(ax, "x")
ax.set_xlabel("Share of rows ÷ share of hours in the study period (log scale)")
ax.set_title("Share of rows against share of hours, by group")
ax.plot([], [], "o", color=DATA, label="within ±25%")
ax.plot([], [], "o", color=REFERENCE, label="outside ±25% (labelled)")
ax.legend(loc="upper right")
save(fig, "ratio_by_group.png")

## 3. Time of day: share of rows vs share of pedestrians

Hours are spread evenly through the day, but pedestrians aren't: night has about a quarter of the rows and very few of the people. An error measured in people per hour would hardly notice night errors, which is why step 2 uses relative error.

In [ ]:
t = slice_rows("time_of_day", list(cfg["time_of_day"]))
x = np.arange(len(t))
w = 0.38

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(x - w / 2, t.row_share, w, color=DATA, edgecolor=SURFACE, linewidth=1, label="Share of rows")
ax.bar(x + w / 2, t.pedestrian_share, w, color=REFERENCE, edgecolor=SURFACE, linewidth=1, label="Share of pedestrians")
night = t.index[t.group == "night"][0]
for dx, v in [(-w / 2, t.row_share[night]), (w / 2, t.pedestrian_share[night])]:
    ax.text(x[night] + dx, v + 0.008, f"{v:.1%}", ha="center", fontsize=9, color=INK)
ax.set_xticks(x, [pretty(g).rsplit(" ", 1)[0] + "\n" + pretty(g).rsplit(" ", 1)[1] for g in t.group])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
grid_on(ax)
ax.set_title(f"Night is {t.row_share[night]:.0%} of rows but {t.pedestrian_share[night]:.1%} of pedestrians")
ax.legend(loc="upper left")
save(fig, "time_of_day_rows_vs_pedestrians.png")

## 4. Coverage by year

Two measures on different scales, so two panels rather than a second y-axis. Until the Wayback fill is in the pipeline, November 2022 to October 2024 has no rows: 2023 is empty (shaded) and 2022 and 2024 are partial.

In [ ]:
yrs = years.set_index("year").reindex(range(years.year.min(), years.year.max() + 1))
gap_years = yrs.index[yrs.n_rows.isna()].tolist()

fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 5.2), sharex=True, gridspec_kw={"hspace": 0.35})
a1.bar(yrs.index, yrs.n_sensors, 0.75, color=DATA, edgecolor=SURFACE, linewidth=1)
a2.bar(yrs.index, yrs.n_rows / 1e6, 0.75, color=DATA, edgecolor=SURFACE, linewidth=1)
a1.set_title("Sensors and rows per year")
a1.set_ylabel("Sensors")
a2.set_ylabel("Rows (millions)")
for ax in (a1, a2):
    grid_on(ax)
    for g in gap_years:
        ax.axvspan(g - 0.45, g + 0.45, color=GRID, alpha=0.8, linewidth=0)
a2.set_xticks(yrs.index[::2])
if gap_years:
    a2.text(0, -0.28, "Shaded: no rows. November 2022 to October 2024 is missing until the Wayback fill is in "
            "the pipeline, so 2022 and 2024 are partial.", transform=a2.transAxes, fontsize=8, color=INK2)
save(fig, "coverage_by_year.png")
print("years with no rows:", gap_years)

## 5. Median count in the cross-tables

Problems often show where two slices meet. One shade per cell, light to dark blue with the median count; the number is printed in each cell. Cells with no data are left blank.

In [ ]:
def heatmap(dim_a, dim_b, order_a, order_b, title, name, figsize):
    c = cross[(cross.dimension_a == dim_a) & (cross.dimension_b == dim_b)
              & ~cross.group_a.isin(["unknown", "no_location"]) & ~cross.group_b.isin(["unknown", "no_location"])]
    m = c.pivot(index="group_a", columns="group_b", values="median_count")
    m = m.reindex(index=[g for g in order_a if g in m.index] if order_a else sorted(m.index),
                  columns=[g for g in order_b if g in m.columns])
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(m.values, cmap=BLUES, aspect="auto")
    vmax = np.nanmax(m.values)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9,
                        color="#ffffff" if v > 0.55 * vmax else INK)
    ax.set_xticks(range(m.shape[1]), [pretty(g) for g in m.columns])
    ax.set_yticks(range(m.shape[0]), [pretty(g) for g in m.index])
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, shrink=0.8)
    cb.set_label("Median people per hour", color=INK2)
    cb.outline.set_visible(False)
    ax.set_title(title)
    save(fig, name)


heatmap("time_of_day", "rain", list(cfg["time_of_day"]), list(cfg["rain_mm"]["labels"]),
        "Median count by time of day and rain", "heatmap_time_of_day_x_rain.png", (7, 4))
precinct_order = slice_rows("precinct").sort_values("n_sensors", ascending=False).group.tolist()
heatmap("precinct", "era", precinct_order, list(cfg["eras"]),
        "Median count by precinct and era",
        "heatmap_precinct_x_era.png", (8, 5))

## 6. Slope: share of sensors vs share of footpath length

Each slope band's share of the located sensors next to its share of the surveyed footpath length (the report's reference), with the ratio at the end of each pair.

In [ ]:
s = slice_rows("slope_band", list(cfg["slope_pct"]["labels"]))
s["sensor_share"] = s.n_sensors / s.n_sensors.sum()
x = np.arange(len(s))
w = 0.38

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w / 2, s.sensor_share, w, color=DATA, edgecolor=SURFACE, linewidth=1, label="Share of sensors")
ax.bar(x + w / 2, s.reference_share, w, color=REFERENCE, edgecolor=SURFACE, linewidth=1, label="Share of footpath length")
for xi, r in zip(x, s.itertuples()):
    ax.text(xi, max(r.sensor_share, r.reference_share) + 0.015, f"×{r.ratio:.2f}", ha="center", fontsize=9, color=INK)
ax.set_xticks(x, [f"{g}\n({n} sensors)" for g, n in zip(s.group, s.n_sensors)])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylim(0, s[["sensor_share", "reference_share"]].max().max() * 1.2)
grid_on(ax)
ax.set_title("Share of sensors against share of footpath length, by slope band")
ax.legend(loc="upper right")
save(fig, "slope_sensors_vs_footpath.png")

## 7. Step 2: the baseline's error per slice group

Each group's WAPE on held-out sensors, with its 95% interval (from resampling the group's sensors), against the overall WAPE (vertical line). Groups marked `worse` in `slice_errors.csv` (the whole interval above the overall error) are orange. `unknown` groups (hours with no weather or calendar yet) are left out.

In [ ]:
DIMS = {   # slice -> heading, in the doc's order
    "time_of_day": "Time of day", "day_type": "Day", "public_holiday": "Public holiday",
    "school_term": "School term", "uni_teaching": "University", "rain": "Rain", "temperature": "Temperature",
    "era": "Era", "afl_day": "AFL match day", "source": "Source", "precinct": "Precinct",
    "holdout_precinct": "Held-out precinct", "volume_tier": "Sensor volume", "amenity_tier": "Amenities nearby",
    "slope_band": "Slope", "missing_hours": "Hours missing", "match_review": "Sensor match",
    "sensor_start": "Sensor start",
}
base = errors[(errors.mitigation == "none") & (errors.group != "unknown")]
ORDER = {"time_of_day": list(cfg["time_of_day"]), "rain": cfg["rain_mm"]["labels"],
         "temperature": cfg["temperature_c"]["labels"], "era": list(cfg["eras"])}   # others: as in the CSV
overall = summary.loc["none", "wape"]

rows, ticks = [], []   # a heading row (None) before each slice's groups
for dim, title in DIMS.items():
    rows.append(None)
    ticks.append(title)
    d = base[base.dimension == dim]
    if dim in ORDER:
        d = d.set_index("group").loc[[g for g in ORDER[dim] if g in set(d.group)]].reset_index()
    for r in d.itertuples():
        rows.append(r)
        ticks.append(f"{pretty(r.group)}  ({r.n_sensors})")
y = np.arange(len(rows))[::-1]

fig, ax = plt.subplots(figsize=(8, 17))
ax.axvline(overall, color=AXIS, linewidth=1.5)
for yi, r in zip(y, rows):
    if r is None:
        continue
    colour = REFERENCE if r.worse else DATA
    ax.plot([r.wape_lo, r.wape_hi], [yi, yi], color=colour, linewidth=2, solid_capstyle="round")
    ax.plot(r.wape, yi, "o", ms=7, color=colour, markeredgecolor=SURFACE, markeredgewidth=1.5)
ax.set_yticks(y, ticks)
for label, r in zip(ax.get_yticklabels(), rows):
    if r is None:
        label.set_fontweight("bold")
        label.set_color(INK)
ax.tick_params(axis="y", length=0)
ax.set_ylim(-1, len(rows))
grid_on(ax, "x")
ax.text(overall, len(rows) - 0.3, f" overall {overall:.2f}", color=INK2, fontsize=9, va="bottom")
ax.set_xlabel("WAPE on held-out sensors (dot), with 95% interval (line)")
ax.set_title("Baseline error by slice group (number of sensors)", pad=18)
ax.plot([], [], "o-", color=DATA, label="not worse")
ax.plot([], [], "o-", color=REFERENCE, label="worse: interval above the overall error")
ax.legend(loc="upper right")
save(fig, "slice_errors_baseline.png", dpi=95)   # tall figure: lower dpi keeps it under ~200 KB

## 8. Step 3: WAPE under each mitigation

The groups in the doc's step 3 table, under the baseline and the three mitigations. The doc's table gives the same numbers.

In [ ]:
MITIGATIONS = {"none": "Baseline", "sensor_equal": "Equal sensors",
               "precinct_to_network": "Precincts to network", "calibrated": "Calibrated"}
COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # validated categorical slots 1-4
STEP3 = [   # (slice, group, label): the rows of the step 3 table in docs/bias/bias_slicing.md
    ("volume_tier", "quiet", "Quiet sensors"), ("precinct", "Southbank", "Southbank"),
    ("amenity_tier", "low", "Low-amenity sensors"), ("slope_band", ">=5%", "Steep paths (≥5%)"),
    ("match_review", "needs_review", "Match needs review"), ("public_holiday", "holiday", "Public holidays"),
    ("era", "covid", "COVID era"), ("time_of_day", "night", "Night"), ("precinct", "Docklands", "Docklands"),
    ("precinct", "Melbourne (Remainder)", "Melbourne (Remainder)"), ("precinct", "CBD", "CBD"),
    ("holdout_precinct", "carlton", "Carlton (held out)"),
    ("holdout_precinct", "north_melbourne", "North Melbourne (held out)"),
]
wape = errors.set_index(["mitigation", "dimension", "group"])["wape"]
labels = ["Overall"] + [label for _, _, label in STEP3]
y = np.arange(len(labels))[::-1]
h = 0.2

fig, ax = plt.subplots(figsize=(8, 10))
for k, (m, name) in enumerate(MITIGATIONS.items()):
    values = [summary.loc[m, "wape"]] + [wape[(m, d, g)] for d, g, _ in STEP3]
    ax.barh(y + (1.5 - k) * h, values, h, color=COLOURS[k], edgecolor=SURFACE, linewidth=1, label=name)
ax.set_yticks(y, labels)
ax.get_yticklabels()[0].set_fontweight("bold")
ax.tick_params(axis="y", length=0)
grid_on(ax, "x")
ax.set_xlabel("WAPE on held-out sensors")
ax.set_title("WAPE by mitigation, for the groups in the step 3 table", pad=30)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4, handlelength=1.2, columnspacing=1.2)
save(fig, "slice_errors_mitigations.png")

## Figures written

In [ ]:
for p in sorted(FIGURES.glob("*.png")):
    print(f"{p.name:<42} {p.stat().st_size / 1e3:6.0f} KB")